In [1]:
import os 
from dotenv import load_dotenv 

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq 

model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=os.getenv("GROQ_API_KEY") 
)

In [4]:
from langchain_core.messages import HumanMessage, AIMessage 

messages = [
    AIMessage(content="Bạn là trợ lý AI"),
    HumanMessage(content="Python là gì?"), 
]

result = model.invoke(messages) 

In [5]:
result.content 

'**Python** là một ngôn ngữ lập trình cấp cao, được thiết kế với mục tiêu tối ưu tính dễ đọc và viết mã. Dưới đây là những điểm chính giúp bạn hiểu rõ hơn về Python:\n\n| # | Nội dung | Mô tả |\n|---|----------|-------|\n| 1 | **Nguồn gốc** | Được Guido van Rossum phát triển tại Bảo tàng Cộng đồng Nghiên cứu (CWI) Hà Lan, xuất bản lần đầu tiên vào năm 1991. |\n| 2 | **Triết lý thiết kế** | *“Code should read like English.”* Python nhấn mạnh cấu trúc mã rõ ràng, giảm thiểu lệnh lặp lại, và khuyến khích viết mã ngắn gọn, dễ bảo trì. |\n| 3 | **Cú pháp** | Dùng dấu phẩy, dấu chấm, và đặc biệt là **định hướng thụt lề** (indentation) để xác định khối lệnh thay vì dấu ngoặc `{}`. |\n| 4 | **Chạy trên máy ảo** | Python là ngôn ngữ *interpreted* (được dịch từng dòng khi chạy). Nó chạy trên *Python Virtual Machine (PVM)*, giúp mã có thể chạy trên nhiều nền tảng (Windows, macOS, Linux, iOS, Android…). |\n| 5 | **Thư viện phong phú** | - **Standard Library**: bao gồm `math`, `datetime`, `json`, `

In [6]:
# Message history/context 
from langchain_community.chat_message_histories import ChatMessageHistory 
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory

In [7]:
store = {} # Đây là dict lưu trữ các cuộc hội thoại theo session (session_ID - conversations) 

def get_session_history(session_id:str) -> BaseChatMessageHistory: 
    if session_id not in store: 
        store[session_id] = ChatMessageHistory() # Khởi tạo session (nếu chưa tồn tại trong dict store)
    return store[session_id]

In [8]:
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [9]:
chat_session_1 = {
    "configurable": {
        "session_id": "session_1" 
    }
}

In [10]:
messages = [
    HumanMessage(content="Xin chào, tôi tên là An, tôi là một giảng viên")
]

response = with_message_history.invoke(
    messages, 
    config=chat_session_1
)

In [11]:
response.content

'Chào An! Rất vui được gặp bạn. Bạn cần hỗ trợ gì hôm nay? Có thể là về dạy học, nghiên cứu, công nghệ, hay bất cứ điều gì khác nhé.'

In [12]:
chat_session_2 = {
    "configurable": {
        "session_id": "session_2" 
    }
}

In [15]:
messages = [
    HumanMessage(content="Tôi là ai, và tôi làm gì?")
]

response = with_message_history.invoke(
    messages, 
    config=chat_session_2
)

In [16]:
response.content

'Bạn là ai, và bạn làm gì? – đó là câu hỏi rất sâu sắc, và câu trả lời thực sự phụ thuộc vào góc độ và mục tiêu của bạn.\n\n| Góc độ | Ví dụ về câu trả lời |\n|--------|------------------------|\n| **Nhân cách** | Bạn có thể là người hướng nội, hướng ngoại, sáng tạo, thực dụng… |\n| **Vai trò xã hội** | Bạn có thể là học sinh, sinh viên, nhân viên, gia đình, bạn bè, đồng nghiệp… |\n| **Sở thích / đam mê** | Bạn thích đọc sách, thể thao, nghệ thuật, công nghệ, du lịch… |\n| **Mục tiêu / ước mơ** | Bạn muốn học một ngôn ngữ, làm việc trong lĩnh vực nào, xây dựng một dự án, hoặc đơn giản là tìm kiếm hạnh phúc? |\n| **Kỹ năng / năng lực** | Bạn có kỹ năng giao tiếp, lập trình, lãnh đạo, tư duy phản biện… |\n\nNếu bạn muốn một câu trả lời cụ thể hơn, bạn có thể chia sẻ thêm về:\n\n1. **Bối cảnh hiện tại** – Bạn đang làm gì, đang học gì, đang sống ở đâu?\n2. **Mục tiêu ngắn hạn / dài hạn** – Bạn muốn đạt được điều gì trong 6 tháng tới, 5 năm tới?\n3. **Sở thích, đam mê** – Những điều gì khiế

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Bạn là trợ lý AI, bạn sẽ trả lời bằng {language}"), 
        MessagesPlaceholder(variable_name="input")
    ]
)

chain = prompt | model

In [19]:
chain_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history, 
    input_messages_key="input"
)

In [20]:
chat_session_new = {
    "configurable": {
        "session_id": "chat_session_new" 
    }
}

In [21]:
response = chain_message_history.invoke(
    {
        "input": [
            AIMessage(content="Bạn là giảng viên chuyên dạy về lập trình, hãy trả lời một cách chuyên nghiệp"),
            HumanMessage(content="Python là gì?")
        ], 
        "language": "Bồ Đào Nha" 
    }, 
    config=chat_session_new
)

In [22]:
response.content

'**Python** é uma linguagem de programação de alto nível, interpretada e de tipagem dinâmica, que se destaca por sua sintaxe clara e legível. Foi criada por Guido van Rossum e lançada em 1991, e desde então tornou‑se uma das linguagens mais populares e versáteis no mundo do desenvolvimento de software.\n\n### Características principais\n\n| Característica | Descrição |\n|----------------|-----------|\n| **Interpretação** | O código é executado linha a linha, facilitando o teste e a depuração. |\n| **Tipagem dinâmica** | Os tipos de dados são inferidos em tempo de execução, reduzindo a necessidade de declarações explícitas. |\n| **Sintaxe legível** | Estruturas de controle, blocos e funções são expressos de forma intuitiva, o que acelera a escrita e a manutenção do código. |\n| **Paradigmas múltiplos** | Suporta programação procedural, orientada a objetos e funcional. |\n| **Bibliotecas padrão extensas** | Inclui módulos para manipulação de arquivos, redes, bases de dados, ciência de da